# Pregnancy Journey Partner Chatbot
## Notebook 03: Retrieval Testing

This notebook evaluates the semantic search performance of the Pregnancy Journey Partner knowledge base.

Objectives:
- Load the processed knowledge base
- Connect to ChromaDB
- Load the embedding model
- Test semantic retrieval
- Evaluate retrieval quality before building the chatbot

In [1]:
from pathlib import Path

import polars as pl
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer

In [2]:
# Define project paths

processed_path = Path("../knowledge_base/processed")

chunks_file = processed_path / "chunks.parquet"
embeddings_file = processed_path / "embeddings.npy"

vector_db_path = "../knowledge_base/vector_db"

In [3]:
# Load processed chunks

chunks_df = pl.read_parquet(chunks_file)

print("Knowledge base loaded successfully.")

print(f"Total chunks: {chunks_df.height}")
print(f"Columns: {chunks_df.columns}")

Knowledge base loaded successfully.
Total chunks: 3596
Columns: ['chunk_id', 'organization', 'document', 'page_number', 'chunk_number', 'text', 'source_id', 'title', 'url']


In [4]:
# Load embeddings

loaded_embeddings = np.load(embeddings_file)

print("Embeddings loaded successfully.")

print("Shape:", loaded_embeddings.shape)

Embeddings loaded successfully.
Shape: (3596, 384)


In [5]:
# Load embedding model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Embedding dimension: 384


C:\Users\Ark of Designs\AppData\Local\Temp\ipykernel_2960\2541707683.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [6]:
# Connect to ChromaDB

chroma_client = chromadb.PersistentClient(
    path=vector_db_path
)

knowledge_collection = chroma_client.get_collection(
    "pregnancy_knowledge"
)

print("Connected to ChromaDB.")

print("Stored chunks:", knowledge_collection.count())

Connected to ChromaDB.
Stored chunks: 3596


In [7]:
# Final verification

assert chunks_df.height == loaded_embeddings.shape[0]
assert chunks_df.height == knowledge_collection.count()

print("Everything loaded successfully.")
print("Knowledge base is ready for retrieval testing.")

Everything loaded successfully.
Knowledge base is ready for retrieval testing.


In [8]:
# Search the knowledge base

def search_knowledge_base(question, n_results=5):
    """
    Search the Pregnancy Journey Partner knowledge base.

    Parameters
    ----------
    question : str
        User's question.

    n_results : int
        Number of chunks to retrieve.

    Returns
    -------
    dict
        ChromaDB search results.
    """

    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = knowledge_collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=n_results
    )

    return results

In [9]:
# Display search results

def display_results(results):

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for i in range(len(documents)):

        print("=" * 80)
        print(f"RESULT {i + 1}")
        print("=" * 80)

        print(f"Organization : {metadatas[i]['organization']}")
        print(f"Document     : {metadatas[i]['document']}")
        print(f"Page         : {metadatas[i]['page_number']}")
        print(f"Distance     : {distances[i]:.4f}")

        print("\nTitle:")
        print(metadatas[i]["title"])

        print("\nSource:")
        print(metadatas[i]["url"])

        print("\nRetrieved Text:\n")

        print(documents[i][:1200])

        print("\n")

In [10]:
question = "What are the danger signs during pregnancy?"

results = search_knowledge_base(question)

display_results(results)

RESULT 1
Organization : WHO
Document     : Who pregnancy, childbirth, postpartum
Page         : 165
Distance     : 0.7339

Title:
Pregnancy Childbirth Postpartum and Newborn Care: A Guide for Essential Practice

Source:
https://www.who.int/publications/b/31363

Retrieved Text:

after birth.
When to seek care for danger signs
Go to hospital or health centre immediately, day or night, DO NOT wait, if any of the following signs:
 Vaginal bleeding has increased.
 Fits.
 Fast or difficult breathing.
 Fever and too weak to get out of bed.
 Severe headaches with blurred vision.
 Calf pain, redness or swelling; shortness of breath or chest pain.
Go to health centre as soon as possible if any of the following signs:
 Swollen, red or tender breasts or nipples.
 Problems urinating, or leaking.
 Increased pain or infection in the perineum.
 Infection in the area of the wound.
 Smelly vaginal discharge.
INFORMATION AND COUNSELLING SHEETS
M4Care for the mother after birth


RES

In [11]:
results = search_knowledge_base(
    "What are the signs that labour is starting?"
)

display_results(results)

RESULT 1
Organization : WHO
Document     : Who pregnancy, childbirth, postpartum
Page         : 72
Distance     : 1.0611

Title:
Pregnancy Childbirth Postpartum and Newborn Care: A Guide for Essential Practice

Source:
https://www.who.int/publications/b/31363

Retrieved Text:

FIRST STAGE OF LABOUR: IN ACTIVE LABOUR
Use this chart when the woman is IN ACTIVE LABOUR, when cervix dilated 4 cm or more.
MONITOR EVERY 30 MINUTES: MONITOR EVERY 4 HOURS:
 For emergency signs, using rapid assessment (RAM) B3-B7 .
 Frequency, intensity and duration of contractions.
 Fetal heart rate D14 .
 Mood and behaviour (distressed, anxious) D6 .
 Cervical dilatation D3 D15 .
 Unless indicated, do not do vaginal examination more frequently than every 4 hours.
 Temperature.
 Pulse B3 .
 Blood pressure D23 .
 Record findings regularly in Labour record and Partograph N4-N6 .
 Record time of rupture of membranes and colour of amniotic fluid.
 Give Supportive care D6-D7 .
 Never leave 

In [12]:
results = search_knowledge_base(
    "What is pre-eclampsia?"
)

display_results(results)

RESULT 1
Organization : WHO
Document     : WHO POLICY PREECLAMPSIA
Page         : 9
Distance     : 0.4545

Title:
WHO Recommendations on Severe Pre-eclampsia Before Term

Source:
https://www.who.int/publications/i/item/9789241550444

Retrieved Text:

maternal
and neonatal health. It is one of the leading
causes of maternal and perinatal mortality and
morbidity worldwide. However, the pathogenesis
of pre-eclampsia is only partially understood,
and it is related to disturbances in placentation
at the beginning of pregnancy, followed by
generalized inflammation and progressive
endothelial damage. There are other uncertainties
too: the diagnosis, screening and management of
pre-eclampsia remains controversial, as does the
classification of its severity.
The 11th revision of the International Classification
of Diseases (ICD-11) describes pre-eclampsia
as a condition characterized by systolic
blood pressure greater than 140 mmHg, and/
or diastolic greater or equal to 90 mmHg on
two occasions

In [13]:
results = search_knowledge_base(
    "What should I eat during pregnancy?"
)

display_results(results)

RESULT 1
Organization : WHO
Document     : Who pregnancy, childbirth, postpartum
Page         : 44
Distance     : 0.6964

Title:
Pregnancy Childbirth Postpartum and Newborn Care: A Guide for Essential Practice

Source:
https://www.who.int/publications/b/31363

Retrieved Text:

the woman to eat a greater amount and variety of healthy foods, such as meat, fish, oils,
nuts, seeds, cereals, beans, vegetables, cheese, milk, to help her feel well and strong (give examples
of types of food and how much to eat).
 Spend more time on nutrition counselling with very thin, adolescent and HIV-infected woman.
 Determine if there are important taboos about foods which are nutritionally important for good
health. Advise the woman against these taboos.
 Talk to family members such as the partner and mother-in-law, to encourage them to help ensure
the woman eats enough and avoids hard physical work.
Advise on self-care during pregnancy
Advise the woman to:
 Take iron tablets F3 .
 Rest and avo

In [14]:
results = search_knowledge_base(
    "What should I do if I have heavy bleeding after birth?"
)

display_results(results)

RESULT 1
Organization : WHO
Document     : WHO BLEEDING AFTER BIRTH
Page         : 8
Distance     : 0.7476

Title:
Bleeding After Birth: Course on Prevention Diagnosis and Treatment of Postpartum Haemorrhage

Source:
https://www.who.int/publications/i/item/9789240115835

Retrieved Text:

Main causes of
bleeding after birth
Tone
Soft uterus
Tissue
Retained placenta
or fragments
Trauma
Genital tears
Thrombin
Coagulopathy
3


RESULT 2
Organization : WHO
Document     : WHO BLEEDING AFTER BIRTH
Page         : 33
Distance     : 0.7983

Title:
Bleeding After Birth: Course on Prevention Diagnosis and Treatment of Postpartum Haemorrhage

Source:
https://www.who.int/publications/i/item/9789240115835

Retrieved Text:

be gentle.
Assess for tears
Ensure good lighting.
Gently wipe away blood and clots from
the vagina and cervix.
Assess the extent of the tears and
related bleeding.
Respond based on your scope of
practice.
Manage tears
Follow local protocols.
Apply firm pressure with a sterile gauze.

In [15]:
question = "What are the danger signs during pregnancy?"

results = search_knowledge_base(question)

print(len(results["documents"][0]))

5


In [16]:
question = "What are the danger signs during pregnancy?"

results = search_knowledge_base(question)

for i, metadata in enumerate(results["metadatas"][0], start=1):
    print(f"Result {i}")
    print("Organization:", metadata["organization"])
    print("Document:", metadata["document"])
    print("Page:", metadata["page_number"])
    print("-" * 50)

Result 1
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 165
--------------------------------------------------
Result 2
Organization: WHO
Document: Who recommendations on Antenatal care for positive pregnancy experience
Page: 92
--------------------------------------------------
Result 3
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 58
--------------------------------------------------
Result 4
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 44
--------------------------------------------------
Result 5
Organization: WHO
Document: Who pregnancy, childbirth, postpartum
Page: 162
--------------------------------------------------
